# vEMDestripe comparison — fixed Original793 demo target

**vEMDestripe: network output → ONE brightness/contrast map to
Original `slice_00793.png` → save final images → read-only evaluation.**
Only vEMDestripe receives this map. Original, EMISAC, FECNet and SSCOR
retain their delivered pixels. The evaluator performs no normalization.

按作者指定，本演示固定向一张 Original 图 `slice_00793.png` 的均值和标准差匹配。
默认源统计仍来自完整 2651 张网络输出；所有切片使用同一映射，再截取 32 张展示。
不逐片拟合，不做几何／空间配准。

**This is a user-selected demonstration target, not the previous paper
reference-pool protocol.** Production inference still defaults to its
recorded original-reference pool; the network and weights are unchanged.
EMISAC remains the verified unchanged-official-source precomputed result.
These 32-section diagnostics are not the full-paper benchmark, and no
paired clean target is claimed for the real Bumbarger images.

## 1. Connect and run / 连接运行

Keep **published_full_volume** to use the complete 2651-section source
statistics and fixed Original793 target. This default works without GPU.
Select **Runtime → Run all**. Optional **live_demo** runs 32 new sections
on GPU and fits their own whole-volume source statistics to the **same**
Original793 target; it is clearly labelled as a different source context.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time, zipfile

GITHUB_OWNER, REPOSITORY = "simple-happy-802", "vEMDestripe"
REPOSITORY_REF = "demo-original-target-v4"
RELEASE_TAG = "demo-reference-v3"  # unchanged input/raw assets; not the v4 calibration result
ASSET_NAME = "vemdestripe_bumbarger_reference_pipeline_v3.zip"
ASSET_SHA256 = "9aa8bdbf82dbbafb4473df73abdcc2dea6a9853c4c6aa135cb4a8809f61ff669"
ASSET_URL = (f"https://github.com/{GITHUB_OWNER}/{REPOSITORY}/releases/"
             f"download/{RELEASE_TAG}/{ASSET_NAME}")
VEM_RUN_MODE = 'published_full_volume'  # or 'live_demo' (GPU, separate 32-section context)
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
RUNTIME_ROOT = Path('/content') if IN_COLAB else Path.cwd()/'original793_demo_runtime'
PROJECT_ROOT = RUNTIME_ROOT/(REPOSITORY+'-original793-v4')
DOWNLOAD_ROOT, RESULT_ROOT = RUNTIME_ROOT/'downloads', RUNTIME_ROOT/'original793_comparison_results'
for folder in (RUNTIME_ROOT, DOWNLOAD_ROOT, RESULT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

def run_command(command, **kwargs):
    result = subprocess.run([str(x) for x in command], text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, **kwargs)
    if result.returncode:
        print(result.stdout[-12000:])
        raise RuntimeError(f'Command failed (exit {result.returncode})')
    return result.stdout
print(f'Runtime: {RUNTIME_ROOT}; Colab: {IN_COLAB}')

In [ ]:
repo_url = f'https://github.com/{GITHUB_OWNER}/{REPOSITORY}.git'
if not (PROJECT_ROOT/'.git').is_dir():
    run_command(['git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF,
                 repo_url, PROJECT_ROOT])
commit = run_command(['git', '-C', PROJECT_ROOT, 'rev-parse', 'HEAD']).strip()
tagged_commit = run_command(['git', '-C', PROJECT_ROOT, 'rev-parse',
                             REPOSITORY_REF+'^{commit}']).strip()
if commit != tagged_commit:
    raise RuntimeError('Existing checkout is not at the versioned demo tag. Use a fresh runtime.')
if run_command(['git', '-C', PROJECT_ROOT, 'status', '--porcelain',
                '--untracked-files=no']).strip():
    raise RuntimeError('The demo source checkout has modified tracked files; use a fresh runtime.')
print('Installing demo dependencies; the existing PyTorch build is retained if compatible.')
run_command([sys.executable, '-m', 'pip', 'install', '-q', '-e',
             f'{PROJECT_ROOT}[evaluation]', 'pandas>=2.2,<3',
             'ipywidgets>=8,<9', 'requests>=2.32,<3', 'h5py>=3.11,<4'])
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT/'colab'))
print(f'Versioned project source: {REPOSITORY_REF}, commit {commit}')

## 2. Download and verify unchanged assets / 校验未改动的数据

The immutable v3 bundle supplies the original input, network-only outputs,
full-source histograms, guidance references and unchanged baseline results.
Its old reference-pool-calibrated images are retained as historical evidence
**but are not used as the current final vEMDestripe output**.
V4 computes and saves a new output with Original793 as the only target.
Keeping this bundle avoids duplicating a large unchanged release asset.

In [ ]:
import requests
from reference_pipeline_demo import sha256_file, safe_extract_bundle, verify_bundle

archive = DOWNLOAD_ROOT/ASSET_NAME
if not archive.is_file():
    partial = archive.with_suffix('.zip.part')
    print('Downloading the original-source demo bundle…')
    with requests.get(ASSET_URL, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()
        with partial.open('wb') as stream:
            for chunk in response.iter_content(1024*1024):
                if chunk:
                    stream.write(chunk)
    if sha256_file(partial) != ASSET_SHA256:
        raise RuntimeError('Downloaded archive checksum mismatch; do not use these results.')
    partial.replace(archive)
if sha256_file(archive) != ASSET_SHA256:
    raise RuntimeError('Cached archive checksum mismatch. Start with a fresh runtime.')
ASSET_ROOT = safe_extract_bundle(archive, DOWNLOAD_ROOT/'official_bundle', ASSET_SHA256)
manifest = verify_bundle(ASSET_ROOT)
print(f'Verified archive: {archive.stat().st_size/1024**2:.1f} MiB')
print('Every bundled file passed its SHA256 check.')

## 3. Original-source EMISAC / 原版 EMISAC

EMISAC receives no new brightness/contrast correction. Its displayed PNGs
are only the declared quantization of the original floating H5. The source,
explicit no-Z-upsampling parameters, old dependency versions and logs are
supplied. Original-source reproduction is documented separately.

In [ ]:
import pandas as pd
from IPython.display import display, Markdown, HTML, Code, FileLink
from official_demo import official_metadata, verify_emisac_pixels

emisac_metadata = official_metadata(ASSET_ROOT)
display(emisac_metadata)
display(verify_emisac_pixels(ASSET_ROOT))
provenance_root = ASSET_ROOT/'provenance'/'emisac_official'
display(Markdown(
    '**EMISAC mode: original-source, legacy-environment precomputed output.** '
    'No Python 3 compatibility result is used in this edition.'))
display(Markdown(f'[Official source and reproducibility instructions]('
                 f'https://github.com/{GITHUB_OWNER}/{REPOSITORY}/blob/{commit}/'
                 'colab/official_emisac/REPRODUCE.md)'))

## 4. Obtain vEMDestripe network output / 网络输出

The released checkpoint uses mean reference fusion and 16×16 block averaging.
The default frozen source was verified against its complete 2651-section
original output. Live mode explicitly produces network-only intermediate
files first; calibration is the next step, not part of the evaluator.

In [ ]:
import torch
checkpoint = PROJECT_ROOT/'checkpoints/vemdestripe_seed42/best.pth'
CHECKPOINT_SHA256 = '9a01cdef1f09716fe87369b057633f9e1278e86e77cbc70eb66b75798709c44a'
if sha256_file(checkpoint) != CHECKPOINT_SHA256:
    raise RuntimeError('Released checkpoint checksum mismatch')
if VEM_RUN_MODE == 'published_full_volume':
    NETWORK_RAW_DIR = ASSET_ROOT/'precomputed/vEMDestripe'
    VEM_SOURCE = 'complete 2651-section network output; demo-only fixed Original793 target; crop after one map'
elif VEM_RUN_MODE == 'live_demo':
    if not torch.cuda.is_available():
        raise RuntimeError('live_demo needs a GPU. Select a GPU or use published_full_volume.')
    NETWORK_RAW_DIR = RESULT_ROOT/'vEMDestripe_network_raw_live'
    NETWORK_RAW_DIR.mkdir(parents=True,exist_ok=True)
    command = [sys.executable, PROJECT_ROOT/'infer.py',
        '--input_dir', ASSET_ROOT/'input', '--output_dir', NETWORK_RAW_DIR,
        '--model', checkpoint, '--reference', ASSET_ROOT/'reference',
        '--num_reference','16','--reference_mode','fused',
        '--reference_preprocess','block_mean','--struct_block_size','16',
        '--reference_selection','uniform','--reference_seed','42',
        '--image_size','1024','--inference_size','1024',
        '--no_sliding_window','--no_z_context','--no_z_postprocess',
        '--no_preview','--disable_progress_bar','--no_photometric_calibration']
    started=time.time(); log=run_command(command)
    (RESULT_ROOT/'network_inference.log').write_text(log,encoding='utf-8')
    VEM_SOURCE = 'live 32-section network output; one map to Original793 using all 32 new outputs; not the full-volume source fit'
    print(f'Live network inference: {(time.time()-started)/60:.1f} min')
else:
    raise ValueError('VEM_RUN_MODE must be published_full_volume or live_demo')
print(VEM_SOURCE)

## 5. Match to Original793, then SAVE / 向指定 Original 匹配后保存

Target: all pixels of **Original `slice_00793.png`**, not a clean label,
not the reference pool, and not each corresponding input slice.

`scale = std(Original793) / std(all network output voxels)`

`bias = mean(Original793) - scale * mean(all network output voxels)`

The same float32 256-level clip → rint → uint8 LUT is used for every
section. Default source statistics cover all 2651 saved outputs; live mode
covers all 32 newly inferred sections. No per-slice source fitting occurs.
Save final PNGs/TIFF and `CALIBRATION.json` before the evaluator reads them.

In [ ]:
from original_target_demo import materialize_original_target_pipeline, verify_exported_raw_files, TARGET_SECTION_ID
TARGET_PATH=ASSET_ROOT/'input'/f'slice_{TARGET_SECTION_ID:05d}.png'
FINAL_VEM_DIR=RESULT_ROOT/('vEMDestripe_original793_final_'+VEM_RUN_MODE)
marker=FINAL_VEM_DIR/'CALIBRATION.json'
if marker.is_file():
    calibration=json.loads(marker.read_text(encoding='utf-8'))
    expected_count=2651 if VEM_RUN_MODE=='published_full_volume' else 32
    if (calibration.get('target_section_id')!=793
        or calibration.get('target_source')!='single_original_slice'
        or calibration['target_file']['sha256']!=sha256_file(TARGET_PATH)
        or calibration['source_statistics']['section_count']!=expected_count):
        raise RuntimeError('Different calibration target/source context. Use a fresh runtime.')
    verify_exported_raw_files(calibration,NETWORK_RAW_DIR)
    for item in [*calibration['output_files'],calibration['output_stack']]:
        if sha256_file(FINAL_VEM_DIR/item['name'])!=item['sha256']:
            raise RuntimeError('Saved final output changed; do not evaluate it.')
else:
    calibration=materialize_original_target_pipeline(ASSET_ROOT,FINAL_VEM_DIR,
        source_dir=NETWORK_RAW_DIR if VEM_RUN_MODE=='live_demo' else None)
display({key:calibration[key] for key in
         ('source_statistics','target_statistics','target_file','scale','bias')})
import matplotlib.pyplot as plt
from PIL import Image
plt.figure(figsize=(5,5))
plt.imshow(Image.open(TARGET_PATH),cmap='gray',vmin=0,vmax=255)
plt.title('Fixed calibration target: Original slice_00793.png')
plt.axis('off');plt.show()
print('One shared map saved for the whole vEM output volume. Evaluator normalization: NONE.')

## 6. Reload and inspect final files / 重新读取最终文件

**Final pipeline outputs** compares vEMDestripe after the fixed Original793
mapping with unchanged delivered baselines. **Network-only diagnostic**
only replaces vEMDestripe's view with its pre-calibration intermediate.
All panels use the same 0–255 display range. XZ/YZ explicitly enlarge Z 6×.
The main table always evaluates the saved final files, independent of the view toggle.

In [ ]:
from evaluation.demo import load_saved_methods
from reference_pipeline_demo import method_directories
from demo_metrics import METHOD_ORDER, load_png_volume, plot_xy, plot_cross_sections, plot_z_profiles
import ipywidgets as widgets
from ipywidgets import interact,IntSlider
FINAL_METHOD_DIRS=method_directories(ASSET_ROOT,FINAL_VEM_DIR)
volumes,section_ids=load_saved_methods(FINAL_METHOD_DIRS,expected_ids=range(777,809))
network_volume,network_ids=load_png_volume(NETWORK_RAW_DIR)
if network_ids!=section_ids or network_volume.shape!=volumes['vEMDestripe'].shape:
    raise RuntimeError('Network and final display windows differ')
network_diagnostic={**volumes,'vEMDestripe':network_volume}
sources={'Original':'delivered input, unchanged',
         'EMISAC':'unchanged official source + legacy environment, precomputed; no new calibration',
         'FECNet':'delivered grayscale adaptation/fine-tuned output; no new calibration',
         'SSCOR':'delivered dataset-specific seed-42 output; no new calibration',
         'vEMDestripe':VEM_SOURCE}
display(pd.DataFrame([{'method':name,'source':sources[name],
                       'upstream_calibration':name=='vEMDestripe'} for name in METHOD_ORDER]))
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
display_mode=widgets.ToggleButtons(options=['Final pipeline outputs','Network-only diagnostic'],
                                   value='Final pipeline outputs',description='Display:')
display(display_mode)
@interact(z=IntSlider(min=0,max=31,value=16,description='Z offset'),mode=display_mode)
def inspect_xy(z,mode):
    print(mode)
    plot_xy(volumes if mode=='Final pipeline outputs' else network_diagnostic,section_ids,z)

In [ ]:
@interact(y=IntSlider(min=32,max=991,step=8,value=512,description='Y'),mode=display_mode)
def inspect_xz(y,mode):
    print(mode)
    plot_cross_sections(volumes if mode=='Final pipeline outputs' else network_diagnostic,
                        axis='xz',position=y,z_ratio=6.0)
@interact(x=IntSlider(min=32,max=991,step=8,value=512,description='X'),mode=display_mode)
def inspect_yz(x,mode):
    print(mode)
    plot_cross_sections(volumes if mode=='Final pipeline outputs' else network_diagnostic,
                        axis='yz',position=x,z_ratio=6.0)
@interact(mode=display_mode)
def inspect_brightness(mode):
    print(mode)
    plot_z_profiles(volumes if mode=='Final pipeline outputs' else network_diagnostic,section_ids)

## 7. Evaluate saved final files / 只评估已保存的最终文件

The evaluator reloads the final PNG files from disk. It neither reads the
reference pool to fit parameters nor normalizes any method. The primary
table includes complete-pipeline vEMDestripe and unchanged baselines.
No all-method affine-matching table is generated.

Native-Z ASID_MAD, local-ASID and Z-HF are continuity diagnostics, not a
guarantee of ultrastructure preservation. NIQE is domain-calibrated and uses
no paired clean target; cross-plane NIQE uses labelled 6× Z enlargement.
No PSNR/SSIM is reported for this real dataset. These are demo-window metrics.

In [ ]:
from evaluation.demo import evaluate_saved_demo
metrics=evaluate_saved_demo(FINAL_METHOD_DIRS,ASSET_ROOT/'bumbarger13_niqe_model.npz',
    RESULT_ROOT/'metrics_final_outputs.csv',xy_samples=8,cross_sections=6,z_ratio=6.0,border=32)
display(metrics.round(4))
assert not metrics['evaluator_photometric_transform'].any()
print('Metrics computed from saved final output pixels; evaluator normalization: NONE.')

## 8. Source code / 源码

vEMDestripe's source includes the network, standalone calibration/export,
and a separate pure evaluator. EMISAC's two scripts are the exact unchanged
files used for its supplied output. FECNet/SSCOR source links retain their
adaptation/training provenance; not all methods run live in Colab.

In [ ]:
from functools import lru_cache
source_specs = {
    'vEMDestripe': {'repo': f'{GITHUB_OWNER}/{REPOSITORY}', 'commit': commit,
        'paths': ['vemdestripe/model.py', 'infer.py', 'vemdestripe/calibration.py', 'calibrate_volume.py', 'colab/original_target_demo.py', 'evaluation/demo.py', 'evaluation/evaluate.py']},
    'EMISAC': {'repo': 'azadis/EMISAC',
        'commit': '7a82a17b7bbb4ca14dff2d1c1a4a018fc8cebaf5',
        'paths': ['EM_normalization.py', 'normalize_partition.py']},
    'FECNet': {'repo': 'KevinJ-Huang/FECNet',
        'commit': 'fc5a9eb224c448d190919558cc87ea7b7447f7f4',
        'paths': ['FastFourierSICE/models/archs/EnhanceN_arch.py', 'FastFourierSICE/test.py']},
    'SSCOR': {'repo': 'AI-XPM-group/SSCOR',
        'commit': '985479cd79bcf1359e3d9ba44bacd5f372eb2e60',
        'paths': ['models/networks.py', 'restore.py', 'train.py']}}

@lru_cache(maxsize=20)
def fetch_source(method, file):
    if method == 'vEMDestripe':
        return (PROJECT_ROOT/file).read_text(encoding='utf-8')
    if method == 'EMISAC':
        return (provenance_root/'source'/file).read_text(encoding='utf-8')
    spec = source_specs[method]
    url = f"https://raw.githubusercontent.com/{spec['repo']}/{spec['commit']}/{file}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.text

method_choice = widgets.Dropdown(options=list(source_specs), description='Method:')
path_choice = widgets.Dropdown(options=source_specs['vEMDestripe']['paths'],
                              description='File:', layout=widgets.Layout(width='650px'))
show_all = widgets.Checkbox(value=False, description='Show complete file')
source_output = widgets.Output()
def render_source(change=None):
    with source_output:
        source_output.clear_output(wait=True)
        method, file = method_choice.value, path_choice.value
        if file not in source_specs[method]['paths']:
            return
        spec = source_specs[method]
        url = f"https://github.com/{spec['repo']}/blob/{spec['commit']}/{file}"
        display(Markdown(f'**{method}** — [complete pinned source]({url})'))
        try:
            lines = fetch_source(method, file).splitlines()
            selected = lines if show_all.value else lines[:200]
            display(Code(chr(10).join(selected), language='python'))
            print(f'Showing {len(selected)}/{len(lines)} lines. Result source: {sources[method]}')
        except Exception as error:
            print(f'Inline source unavailable: {error}. Use the pinned source link above.')
def change_method(change):
    path_choice.options = source_specs[change['new']]['paths']
    render_source()
method_choice.observe(change_method, names='value')
path_choice.observe(render_source, names='value')
show_all.observe(render_source, names='value')
display(widgets.VBox([method_choice, path_choice, show_all, source_output]))
render_source()

## 9. Export / 下载

Exported final outputs are the same files that were evaluated. Network-only
intermediates and the calibration record remain separate. EMISAC raw H5,
original source/GPL, source metadata, data attribution and metrics are retained.
Re-run the source/integrity checks below; display changes cannot alter inputs.

In [ ]:
verify_bundle(ASSET_ROOT)
runtime_provenance={'project_commit':commit,'version_tag':REPOSITORY_REF,
    'asset_sha256':ASSET_SHA256,'local_z_half_open':[777,809],
    'vEMDestripe_mode':VEM_RUN_MODE,'sources':sources,
    'calibrated_methods':['vEMDestripe'],'evaluation_normalization':False,
    'evaluation_input':'saved final PNG files','no_paired_clean_target':True,
    'calibration_source_sections':calibration['source_statistics']['section_count'],
    'calibration_target':'single Original slice_00793.png',
    'target_section_id':793,'target_sha256':sha256_file(TARGET_PATH),
    'demo_target_override':True,'production_reference_default_changed':False}
(RESULT_ROOT/'runtime_provenance.json').write_text(json.dumps(runtime_provenance,indent=2),encoding='utf-8')
for method,folder in FINAL_METHOD_DIRS.items():
    destination=RESULT_ROOT/'final_outputs'/method
    destination.mkdir(parents=True,exist_ok=True)
    for path in Path(folder).glob('slice_*.png'):
        shutil.copy2(path,destination/path.name)
shutil.copytree(NETWORK_RAW_DIR,RESULT_ROOT/'network_only_vEMDestripe',dirs_exist_ok=True)
shutil.copytree(ASSET_ROOT/'calibration',RESULT_ROOT/'historical_v3_source_provenance',dirs_exist_ok=True)
shutil.copytree(provenance_root,RESULT_ROOT/'emisac_official_provenance',dirs_exist_ok=True)
for source in (ASSET_ROOT/'DATA_LICENSE.txt',PROJECT_ROOT/'colab/DATA_LICENSE.md',
               PROJECT_ROOT/'colab/THIRD_PARTY_METHODS.md'):
    if source.is_file():shutil.copy2(source,RESULT_ROOT/source.name)
shutil.copy2(ASSET_ROOT/'manifest.json',RESULT_ROOT/'bundle_manifest.json')
shutil.copy2(TARGET_PATH,RESULT_ROOT/'calibration_target_Original_slice_00793.png')
export_zip=Path(shutil.make_archive(str(RUNTIME_ROOT/'vemdestripe_original793_demo_results'),'zip',RESULT_ROOT))
print(f'Exported {export_zip.name} ({export_zip.stat().st_size/1024**2:.1f} MiB)')
if IN_COLAB:
    from google.colab import files
    button=widgets.Button(description='Download results ZIP',icon='download')
    button.on_click(lambda _:files.download(str(export_zip)))
    display(button)
else:display(FileLink(str(export_zip)))